# 🚀 50M Multilingual GPT - Bengali + English + Math (Full Scratch Training)
### ⚡ হাইলাইট ও স্পেসিফিকেশন:
- **ভাষা ও জ্ঞান:** বাংলা (৭০%) + ইংরেজি (২০%) + গণিত ও সাধারণ পাটিগণিত (১০%)
- **প্যারামিটার:** ~54.3 Million (১০০% আনফ্রোজেন, পুরো মডেল স্ক্র্যাচ থেকে শিখবে)
- **কনটেক্সট লেন্থ:** 512 Tokens (~৩০০-৩৫০ শব্দ ধারণক্ষমতা)
- **ভোকাবুলারি:** 10,000 (বাংলা বর্ণমালা, ইংরেজি A-Z/a-z, গাণিতিক চিহ্ন `+ - * / = < > % ^ √`)
- **ইপক ও স্টেপ:** ২ থেকে ৩ ইপক (Epoc to Step ম্যাথমেটিক্যাল ক্যালকুলেশন দ্বারা নির্ধারিত)
- **হার্ডওয়্যার:** Colab Free T4 GPU (~1.0 GB VRAM, 14 GB মেমরি নিরাপদ)

In [ ]:
# Step 1: GPU চেক করুন (NVIDIA T4 নিশ্চিত করুন)
!nvidia-smi

In [ ]:
# Step 2: Google Drive মাউন্ট করুন (চেকপয়েন্ট সেভ করার জন্য)
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/bengali_gpt_50m_checkpoints'
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print(f"✓ Drive Checkpoint ডিরেক্টরি প্রস্তুত: {DRIVE_CHECKPOINT_DIR}")

In [ ]:
# Step 3: রিপোজিটরি ক্লোন ও লাইব্রেরি ইনস্টল করুন
import os

%cd /content
!rm -rf ss_100m
!git clone https://github.com/kajshikhi49-afk/ss_100m.git

%cd /content/ss_100m/ss_50million
!pip install -q tokenizers torch numpy datasets pyarrow
print("✓ এনভায়রনমেন্ট ও ডিপেনডেন্সি প্রস্তুত!")

In [ ]:
# Step 4: 📥 মাল্টিলিঙ্গুয়াল ডেটা ডাউনলোড (বাংলা + ইংরেজি + পাটিগণিত/ম্যাথ)
# মোট ২.৫ লাখ লাইনের সমন্বিত কর্পাস তৈরি
import os
import re
import random
from datasets import load_dataset

corpus_file = 'data/corpus.txt'
os.makedirs('data', exist_ok=True)

print("⚡ মাল্টিলিঙ্গুয়াল ডেটাসেট ডাউনলোড ও প্রসেসিং শুরু হচ্ছে...")
lines = []

# ১. বাংলা উইকিপিডিয়া (৭০% = ১,৭৫,০০০ লাইন)
print("  -> ১. বাংলা উইকিপিডিয়া সংগ্রহ হচ্ছে (১,৭৫,০০০ লাইন)...")
wiki_bn = load_dataset('wikimedia/wikipedia', '20231101.bn', split='train', streaming=True)
bn_count = 0
for item in wiki_bn:
    for p in item.get('text', '').split('\n'):
        p = p.strip()
        if len(p) >= 30 and re.search(r'[\u0980-\u09FF]', p):
            lines.append(p)
            bn_count += 1
            if bn_count >= 175000: break
    if bn_count >= 175000: break
print(f"     ✓ সংগৃহীত বাংলা লাইন: {bn_count:,}")

# ২. ইংরেজি উইকিপিডিয়া (২০% = ৫০,০০০ লাইন)
print("  -> ২. ইংরেজি উইকিপিডিয়া সংগ্রহ হচ্ছে (৫০,০০০ লাইন)...")
wiki_en = load_dataset('wikimedia/wikipedia', '20231101.en', split='train', streaming=True)
en_count = 0
for item in wiki_en:
    for p in item.get('text', '').split('\n'):
        p = p.strip()
        if len(p) >= 35 and re.search(r'[a-zA-Z]', p):
            lines.append(p)
            en_count += 1
            if en_count >= 50000: break
    if en_count >= 50000: break
print(f"     ✓ সংগৃহীত ইংরেজি লাইন: {en_count:,}")

# ৩. সাধারণ গণিত ও পাটিগণিত সমস্যা (১০% = ২৫,০০০ লাইন)
print("  -> ৩. সাধারণ গণিত ও পাটিগণিত ডেটা তৈরি হচ্ছে (২৫,০০০ লাইন)...")
random.seed(42)
math_lines = []
for _ in range(12500):
    a, b = random.randint(2, 500), random.randint(2, 500)
    # বাংলা পাটিগণিত
    math_lines.append(f"সমস্যা: {a} এর সাথে {b} যোগ করলে কত হয়? সমাধান: {a} + {b} = {a + b}।")
    math_lines.append(f"প্রশ্ন: {a} থেকে {b} বিয়োগ করলে বিয়োগফল কত? উত্তর: {a} - {b} = {a - b}।")
    # ইংরেজি গণিত
    m1, m2 = random.randint(2, 99), random.randint(2, 50)
    math_lines.append(f"Problem: What is {m1} multiplied by {m2}? Solution: {m1} * {m2} = {m1 * m2}.")
    d1, d2 = random.randint(2, 50), random.randint(2, 20)
    math_lines.append(f"Math: If we divide {d1 * d2} by {d2}, the quotient is {d1}.")

lines.extend(math_lines[:25000])
print(f"     ✓ সংগৃহীত গণিত লাইন: {len(math_lines[:25000]):,}")

print(f"✓ সর্বমোট সংগৃহীত লাইন: {len(lines):,} (বাংলা + ইংরেজি + গণিত)")

# ৪. সম্পূর্ণ ডেটা Shuffle (এলোমেলো) করা যাতে তিন বিষয়ের সুষম মিশ্রণ ঘটে
print("⚡ ডেটা এলোমেলো (Shuffle) করা হচ্ছে...")
random.shuffle(lines)

with open(corpus_file, 'w', encoding='utf-8') as f:
    for l in lines:
        f.write(l + '\n')

# পুরোনো বাইনারি ক্যাশ থাকলে ডিলিট
if os.path.exists('data/corpus_tokens.bin'):
    os.remove('data/corpus_tokens.bin')

print(f"✓ সমন্বিত corpus.txt ফাইল তৈরি সম্পন্ন: {os.path.getsize(corpus_file)/(1024*1024):.1f} MB")

In [ ]:
# Step 5: ⚡ সমন্বিত কর্পাস থেকে মাল্টিলিঙ্গুয়াল 10,000 Vocab BPE টোকেনাইজার তৈরি
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

special_tokens = ['<PAD>', '<UNK>', '<BOS>', '<EOS>', '<|system|>', '<|user|>', '<|assistant|>', '<|math|>']
tok = Tokenizer(models.BPE(unk_token='<UNK>'))
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False)
tok.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=10000,
    special_tokens=special_tokens,
    min_frequency=2,
    show_progress=True
)

print("⚡ মাল্টিলিঙ্গুয়াল টোকেনাইজার ট্রেনিং শুরু হচ্ছে...")
tok.train(['data/corpus.txt'], trainer)
tok.save('tokenizer.json')
print(f"✓ টোকেনাইজার সম্পন্ন! Vocab Size: {tok.get_vocab_size():,}")

# ৩টি ডোমেইনের নমুনা টেস্ট:
tests = [
    "বাংলা: কৃত্রিম বুদ্ধিমত্তা হলো ভবিষ্যৎ প্রযুক্তি।",
    "English: Artificial intelligence is transforming the digital world.",
    "Math: 25 * 4 = 100 and x + 15 = 45 implies x = 30."
]
for t in tests:
    enc = tok.encode(t)
    print(f"  {t} -> {len(enc.ids)} tokens")

In [ ]:
# Step 6: 📐 Epoch to Step গাণিতিক হিসাব ও কনফিগারেশন যাচাই
from src.config import GPTConfig
from src.model import BengaliGPT as GPT
from src.dataset import BengaliDataset
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("tokenizer.json")
dataset = BengaliDataset(corpus_path="data/corpus.txt", tokenizer=tokenizer, block_size=GPTConfig.block_size, split_ratio=0.9)

# --- গাণিতিক হিসাব (Epoch to Step Formula) ---
# 1 Batch এ মোট টোকেন = batch_size * gradient_accumulation_steps * block_size
tokens_per_step = GPTConfig.batch_size * GPTConfig.gradient_accumulation_steps * GPTConfig.block_size
tokens_per_epoch = dataset.train_len
steps_per_epoch = tokens_per_epoch // tokens_per_step

# আপনি চেয়েছেন ২ থেকে ৩ ইপক (আমরা আদর্শ ২.৫ ইপক সেট করছি):
TARGET_EPOCHS = 2.5
TOTAL_STEPS = int(steps_per_epoch * TARGET_EPOCHS)

print("=" * 60)
print("📊 ইপক থেকে স্টেপ (Epoch to Step) গাণিতিক রূপান্তর:")
print(f"  - মোট ট্রেনিং টোকেন: {dataset.train_len:,}")
print(f"  - প্রতি স্টেপে প্রসেসকৃত টোকেন: {tokens_per_step:,} ({GPTConfig.batch_size}x{GPTConfig.gradient_accumulation_steps} x {GPTConfig.block_size})")
print(f"  - ১ ইপক (1 Epoch) = {steps_per_epoch:,} স্টেপস")
print(f"  - লক্ষ্যমাত্রা ইপক: {TARGET_EPOCHS} Epochs")
print(f"  - মোট নির্ধারিত স্টেপ (max_iters): {TOTAL_STEPS:,} স্টেপস")
print("=" * 60)

GPTConfig.max_iters = TOTAL_STEPS

In [ ]:
# Step 7: 🚀 প্রোডাকশন প্রি-ট্রেনিং (বাংলা + ইংরেজি + ম্যাথ)
import os
import sys
import time
import math
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from src.config import GPTConfig
from src.model import BengaliGPT as GPT

# ১. ডিভাইস ও সর্বোচ্চ GPU স্পিড ফ্ল্যাগ
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# ২. মডেল ইনিশিয়ালাইজেশন (১০০% আনফ্রোজেন)
raw_model = GPT(GPTConfig).to(device)
total_params = sum(p.numel() for p in raw_model.parameters())
print(f"✓ মডেল সফলভাবে তৈরি হয়েছে! মোট প্যারামিটার: {total_params/1e6:.2f}M (১০০% আনফ্রোজেন)")

try:
    model = torch.compile(raw_model)
    print("✓ torch.compile সক্রিয় (সর্বোচ্চ ট্রেনিং স্পিড)!")
except Exception:
    model = raw_model

# ৩. অপটিমাইজার ও শিডিউলার (Cosine Warmup)
optimizer = torch.optim.AdamW(raw_model.parameters(), lr=GPTConfig.learning_rate, betas=(0.9, 0.95), weight_decay=0.1)
scaler = GradScaler()

def get_lr(it, max_iters, warmup_iters=250, lr=3e-4, min_lr=3e-5):
    if it < warmup_iters:
        return lr * it / warmup_iters
    if it > max_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (lr - min_lr)

@torch.no_grad()
def estimate_val_loss(eval_iters=15):
    raw_model.eval()
    losses = []
    for _ in range(eval_iters):
        x, y = dataset.get_batch('val', batch_size=GPTConfig.batch_size, device=device)
        with autocast(dtype=torch.float16):
            _, loss = raw_model(x, y)
        losses.append(loss.item())
    raw_model.train()
    return sum(losses) / len(losses)

# ৪. ট্রেনিং লুপ
print("=" * 65)
print(f"🔥 মাল্টিলিঙ্গুয়াল স্ক্র্যাচ ট্রেনিং শুরু হচ্ছে ({GPTConfig.max_iters:,} স্টেপস | ২.৫ ইপক)... ")
print("=" * 65)

start_time = time.time()
model.train()
optimizer.zero_grad(set_to_none=True)

for step in range(1, GPTConfig.max_iters + 1):
    current_lr = get_lr(step, max_iters=GPTConfig.max_iters)
    for param_group in optimizer.param_groups:
        param_group['lr'] = current_lr

    accum_loss = 0.0
    for micro_step in range(GPTConfig.gradient_accumulation_steps):
        x, y = dataset.get_batch('train', batch_size=GPTConfig.batch_size, device=device)
        with autocast(dtype=torch.float16):
            logits, loss = model(x, y)
            loss = loss / GPTConfig.gradient_accumulation_steps
        scaler.scale(loss).backward()
        accum_loss += loss.item()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(raw_model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    # বর্তমান ইপক প্রোগ্রেস
    current_epoch = (step * tokens_per_step) / tokens_per_epoch

    # লগ এবং ভ্যালিডেশন লস হিসাব (প্রতি ২৫০ স্টেপে)
    if step % 250 == 0 or step == 1:
        elapsed = time.time() - start_time
        speed = step / elapsed if elapsed > 0 else 0
        val_loss = estimate_val_loss(eval_iters=15)
        print(f"Step {step:4d}/{GPTConfig.max_iters} (Epoch {current_epoch:.2f}) | Train: {accum_loss:.4f} | Val: {val_loss:.4f} | Speed: {speed:.2f} it/s")

    # প্রতি ৫০০ স্টেপ পরপর Google Drive-এ সেভ
    if step % GPTConfig.save_interval == 0 or step == GPTConfig.max_iters:
        ckpt_path = os.path.join(DRIVE_CHECKPOINT_DIR, f"bengali_gpt_50m_step_{step}.pt")
        torch.save({
            'step': step,
            'epoch': current_epoch,
            'model_state_dict': raw_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': accum_loss,
            'config': GPTConfig
        }, ckpt_path)
        print(f"💾 [SAVED] চেকপয়েন্ট ড্রাইভে সেভ হয়েছে: {ckpt_path}")

print("🎉 ৫০M মাল্টিলিঙ্গুয়াল মডেলের ফুল স্ক্র্যাচ ট্রেনিং সম্পন্ন হয়েছে!")

In [ ]:
# Step 8: 💬 বাংলা, ইংরেজি ও গণিত প্রম্পট দিয়ে লাইভ টেস্ট করুন!
eval_model = raw_model if 'raw_model' in locals() else model
eval_model.eval()

test_prompts = [
    "বাংলা: কৃত্রিম বুদ্ধিমত্তা হলো",
    "English: The future of artificial intelligence is",
    "Math: Problem: If x + 10 = 25, then x ="
]

for prompt in test_prompts:
    enc = tokenizer.encode(prompt)
    ids = enc.ids if hasattr(enc, 'ids') else enc
    input_tensor = torch.tensor([ids], dtype=torch.long, device=device)
    
    with torch.no_grad():
        out = eval_model.generate(
            input_tensor,
            max_new_tokens=100,
            temperature=0.7,
            top_k=40,
            repetition_penalty=1.25
        )
    
    reply = tokenizer.decode(out[0].cpu().tolist())
    print("=" * 60)
    print(f"প্রম্পট: {prompt}")
    print(f"উত্তর: {reply}")